# Alert → dashboard: a 45-minute live engineering walkthrough

For interviewers evaluating Robert’s work. Fill in your actual contribution and decision history; no authorship is inferred. The challenge brief remains unknown.

This notebook is a transparent alternative client plus an operator runbook, **not a recording of live success**. The Pi paired-conversation demonstration remains separate. The engine reference is [fe5b613](https://github.com/r33drichards/mcp-js/commit/fe5b613db0afcc9cf20d7ca1ef659c914a0b5008). Open beside [slides.html](slides.html) and [presenter-guide.md](presenter-guide.md).

| Minutes | Activity |
|---|---|
|00–03|Problem, contribution, launch the investigation|
|03–12|Explain actual Compose while it runs|
|12–18|Single-root dashboard preview and policy|
|18–28|Paired Pi/filesystem forks, merge, replacement publication|
|28–35|Three-voter failure and recovery|
|35–39|Tradeoffs and evidence|
|39–45|Q&A|

**Run All is offline by default.** Controls render but never execute actions until clicked and explicitly confirmed. Model execution additionally requires prior provider/budget approval. No credentials belong in notebook cells or saved outputs. Set BEDROCK_MODEL_ID and AWS_REGION only after the runtime lane confirms tool-use support and pricing; set BEDROCK_TOOL_USE_VERIFIED=1 after that check. AWS credentials come from the approved SDK provider chain (prefer the scoped instance role), never copied root credentials. Clear live outputs before sharing. Full Astronomy/Pi workflow remains runtime-unverified here.

In [ ]:
from pathlib import Path
import json, os, sys, subprocess
from IPython.display import display, Markdown, SVG
HERE = Path.cwd().resolve()
if not (HERE / 'demo.py').exists():
    HERE = HERE / 'demos/interview-walkthrough'
assert (HERE / 'demo.py').is_file(), 'Open from the repo root or demo directory'
sys.path.insert(0, str(HERE))
from notebook_support import panel
print('Notebook source directory:', HERE.name)
print('No Docker or model calls have been made.')

## Offline preflight

Install `notebook-requirements.txt` in a dedicated environment outside the repository, then open this notebook with that kernel. This notebook does not install software. Live startup uses README/launch.sh on the approved rehearsal machine; it is not part of Run All.

Do not reseed or run `demo.py test` mid-rehearsal: seed replaces demo.json and can leave existing pair bindings behind. Prepare a fresh owned rehearsal state, verify healthy checkout with the operator fault off, then inject the authorized disposable fault once and verify actual metrics/logs.

In [ ]:
slides = json.loads((HERE / 'slides.json').read_text())
end = 0
for slide in slides:
    start, stop = map(int, slide['clock'].split('–'))
    assert start == end and stop > start
    end = stop
assert end == 45 and slides[-1]['clock'] == '39–45'
print('Verified 45 minutes, including six minutes Q&A.')
print(subprocess.check_output(['git','rev-parse','HEAD'], cwd=HERE, text=True).strip())

In [ ]:
# Preparation only, once, before investigation. Never reseed an active rehearsal.
panel(['seed'])

## 00–03 · Launch a bounded investigation

**What:** paste CheckoutErrors for checkout as untrusted evidence. **How:** the client exposes only bound `branch_info` and `run_js` MCP tools. **Why:** model text does not select arbitrary hosts, branch roots or shell commands.

Use the complete canonical prompt below, not an abbreviated prompt. Only synthetic local telemetry may be sent to the approved provider. Amazon Bedrock Converse is the educational adapter. The AWS runtime lane must verify the cheapest viable tool-capable model and region; no model price or availability is assumed here. Pi remains the separately supported client.

In [ ]:
PROMPT = (HERE / 'pi/agent-prompt.md').read_text()
display(Markdown(PROMPT))
ALERT = json.loads((HERE / 'example-alert.json').read_text())
assert ALERT['labels'] == {'alertname':'CheckoutErrors', 'service':'checkout'}
print(json.dumps(ALERT, indent=2))

### Readable agent loop — definitions only

The real [Bedrock Converse protocol](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_Converse.html) returns `toolUse` blocks; this loop sends `toolResult` blocks back by toolUseId. The async aiobotocore client preserves cancellation without leaving a blocking SDK thread running. The [MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk/tree/v1.30.0) discovers server schemas and executes calls. Installed versions are pinned in notebook-requirements.txt.

Bounded to six model turns, 1,024 output tokens per turn, one tool request per turn, 16 KiB (Nova) or 32 KiB (CountTokens mode) per tool result, mode-specific full-request byte limits and 90 seconds per request. Token usage is logged each turn and cumulatively; these bounds are not a dollar-budget guarantee. Stop cancels the task; it does not roll back completed external effects. A model's final answer does not prove dashboard success: inspect root, file, publication receipt and actual plotted samples. Tool failures and uncertain timeouts stop without automatic retries. This loop stages evidence; operator publication remains outside its tools.

**Budget modes (explicit, no automatic fallback):**

- `BEDROCK_BUDGET_MODE=nova-request-bounds` is permitted only with `BEDROCK_MODEL_ID=us.amazon.nova-micro-v1:0`. The runtime lane verified CountTokens is unsupported for this model. Hard local bounds: 32,768 serialized UTF-8 request bytes per call, 98,304 cumulative submitted request bytes per run, 16,384 serialized bytes per tool result, at most six inference attempts and 1,024 requested output tokens each. The measured full SDK arguments include system instructions, schemas, history and results; HTTP headers are excluded. Oversized content stops rather than truncates. **Bytes are not tokens; exact input token usage is unknown before inference.** Actual provider token usage is logged afterward.
- Default `count-tokens` mode retains the exact full-input counting gate for supported models: 8,192 cumulative input tokens per run. Counting errors fail closed; the code never silently switches budgeting mode or model.

**Conservative spending planning for Nova:** [AWS's Nova Micro card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-amazon-nova-micro.html) documents a 128K context window. Reserving 131,072 input tokens per accepted attempt is a deliberately loose service-context bound, **not an estimate derived from request bytes**. With six attempts and no retries, budget for up to 786,432 input tokens plus 6,144 output tokens at the runtime-verified rates: `786432 × input_USD_per_million / 1e6 + 6144 × output_USD_per_million / 1e6`. This conditional ceiling assumes the published model limits and ordinary token-billed Converse, excludes infrastructure/taxes/other calls, and is not an account-wide dollar cap. Actual use should be lower, but no smaller token ceiling is claimed. IAM/model invocation approval remains separate.

SDK retries are disabled (`total_max_attempts=1`). Dispatches that time out or are cancelled still consume an attempt and byte reservation; no refund or automatic retry. A fresh manually approved run starts a separate budget. The 8,192-character alert intake limit is unrelated to token counting.


In [ ]:
import asyncio

async def investigation_loop(model, session, alert, prompt, emit=print, max_steps=6):
    assert alert.get('labels', {}).get('service') == 'checkout'
    assert alert.get('labels', {}).get('alertname') == 'CheckoutErrors'
    if len(json.dumps(alert)) > 8192: raise ValueError('Alert too large')
    if not 1 <= max_steps <= 6: raise ValueError('Maximum six model turns')
    selected=[t for t in (await session.list_tools()).tools if t.name in {'branch_info','run_js'}]
    assert {t.name for t in selected} == {'branch_info','run_js'}
    tools=[{'toolSpec':{'name':t.name,'description':t.description or t.name,
                       'inputSchema':{'json':t.inputSchema}}} for t in selected]
    messages=[{'role':'user','content':[{'text':json.dumps(alert)}]}]
    totals={'inputTokens':0,'outputTokens':0,'totalTokens':0}
    model_id=os.environ['BEDROCK_MODEL_ID']
    reserved_input_tokens=0
    input_budget=8192  # Cumulative input tokens across the entire run, not alert characters.
    inference_attempts=0
    budget_mode=os.environ.get('BEDROCK_BUDGET_MODE','count-tokens')
    if budget_mode not in {'count-tokens','nova-request-bounds'}:
        raise ValueError('Unknown budget mode')
    if budget_mode=='nova-request-bounds' and model_id!='us.amazon.nova-micro-v1:0':
        raise ValueError('Nova request-bound mode is scoped to the verified runtime model ID')
    submitted_request_bytes=0
    # Hard local size/attempt bounds, NOT a conversion from bytes to tokens.
    request_byte_limit=32768; run_request_byte_limit=98304
    tool_byte_limit=16384 if budget_mode=='nova-request-bounds' else 32768
    emit({'provider':'Bedrock Converse','model':os.environ['BEDROCK_MODEL_ID'],
          'input':alert,'tools':[t['toolSpec']['name'] for t in tools]})
    for step in range(max_steps):
        converse_input={'system':[{'text':prompt+' Request at most one tool call per response.'}],
                        'messages':messages,'toolConfig':{'tools':tools}}
        request={'modelId':model_id,**converse_input,'inferenceConfig':{'maxTokens':1024}}
        # Measure default JSON UTF-8 serialization of the complete SDK arguments,
        # including system/history/tools/results; transport headers are not counted.
        request_bytes=len(json.dumps(request,ensure_ascii=True).encode('utf-8'))
        tokens=None
        if budget_mode=='nova-request-bounds':
            if request_bytes>request_byte_limit:
                raise RuntimeError('Full request byte limit exceeded; no inference sent')
            if submitted_request_bytes+request_bytes>run_request_byte_limit:
                raise RuntimeError('Cumulative request byte limit exceeded; no inference sent')
            emit({'budget_mode':budget_mode,'request_bytes':request_bytes,
                  'input_tokens_before_call':None,'token_count_status':'unknown: CountTokens unsupported',
                  'worst_case_context_tokens_per_attempt':131072})
        else:
            if request_bytes>120000:
                raise RuntimeError('Full input byte safety limit reached (not a token estimate)')
            counted=await asyncio.wait_for(model.count_tokens(
                modelId=model_id,input={'converse':converse_input}),timeout=30)
            tokens=counted.get('inputTokens')
            if type(tokens) is not int or tokens<0:
                raise RuntimeError('Invalid CountTokens response; no inference sent')
            emit({'step':step+1,'counted_input_tokens':tokens,
                  'reserved_input_tokens':reserved_input_tokens,'run_input_budget':input_budget})
            if reserved_input_tokens+tokens>input_budget:
                raise RuntimeError('Cumulative input token budget exceeded; no inference sent')
            reserved_input_tokens+=tokens
        # Reserve before dispatch, including uncertain/cancelled requests. No retries.
        submitted_request_bytes+=request_bytes;inference_attempts+=1
        emit({'inference_attempt':inference_attempts,'budget_mode':budget_mode,
              'reserved_input_tokens':reserved_input_tokens if tokens is not None else None,
              'submitted_request_bytes':submitted_request_bytes,'max_output_tokens':1024})
        response=await asyncio.wait_for(model.converse(**request),timeout=90)
        usage=response.get('usage',{})
        if tokens is not None and usage.get('inputTokens',tokens)>tokens:
            raise RuntimeError('Usage exceeded CountTokens reservation; stop and reconcile billing')
        for key in totals: totals[key]+=usage.get(key,0)
        emit({'step':step+1,'usage':usage,'cumulative_usage':dict(totals),
              'stopReason':response.get('stopReason')})
        message=response['output']['message'];messages.append(message)
        calls=[block['toolUse'] for block in message['content'] if 'toolUse' in block]
        if not calls:
            emit({'findings':message,'verification':'operator checks still required'})
            return messages
        if response.get('stopReason')!='tool_use' or len(calls)!=1:
            raise RuntimeError('Expected one complete tool request; no tool executed')
        call=calls[0];name=call['name'];args=call['input']
        if name not in {'branch_info','run_js'}: raise ValueError('Tool not allowed')
        if name=='run_js' and (set(args)!={'code'} or not isinstance(args['code'],str)):
            raise ValueError('Only bound code allowed; no branch/session override')
        if name=='branch_info' and args: raise ValueError('branch_info takes no arguments')
        emit({'step':step+1,'tool':name,'arguments':args})
        result=await asyncio.wait_for(session.call_tool(name,arguments=args),timeout=90)
        payload=result.model_dump(mode='json',exclude_none=True)
        if len(json.dumps(payload,ensure_ascii=True).encode('utf-8'))>tool_byte_limit: raise RuntimeError('Tool result too large; inspect saved evidence')
        emit({'tool_result':payload})
        messages.append({'role':'user','content':[{'toolResult':{
            'toolUseId':call['toolUseId'],'content':[{'json':payload}],
            'status':'error' if result.isError else 'success'}}]})
        if result.isError: raise RuntimeError('MCP error; inspect evidence before any retry')
    emit({'stop':'Step budget reached; no completion claimed','cumulative_usage':totals})
    return messages

async def live_investigation(alert, emit=print):
    import httpx
    from aiobotocore.session import get_session
    from aiobotocore.config import AioConfig
    from mcp import ClientSession
    from mcp.client.streamable_http import streamable_http_client
    for key in ['AWS_REGION','BEDROCK_MODEL_ID','MCP_JS_DEMO_TOKEN','MCP_JS_DEMO_SESSION']:
        if not os.environ.get(key): raise RuntimeError('Missing approved setup: '+key)
    if os.environ.get('BEDROCK_TOOL_USE_VERIFIED')!='1':
        raise RuntimeError('Runtime lane must verify selected model tool-use capability first')
    import re
    sid=os.environ['MCP_JS_DEMO_SESSION']
    if not re.fullmatch(r'[A-Za-z0-9-]{1,100}',sid): raise ValueError('Invalid bound session ID')
    url='http://127.0.0.1:38080/mcp/'+sid
    from datetime import datetime,timezone,timedelta
    end=datetime.now(timezone.utc).replace(microsecond=0);start=end-timedelta(minutes=5)
    prompt=PROMPT+' Use this same UTC window for metrics and logs: '+start.isoformat()+' to '+end.isoformat()+'.'
    config=AioConfig(connect_timeout=10,read_timeout=80,retries={'total_max_attempts':1})
    # Approved AWS provider chain only; no credential values copied into the notebook.
    async with get_session().create_client('bedrock-runtime',region_name=os.environ['AWS_REGION'],config=config) as model:
        async with httpx.AsyncClient(headers={'X-Demo-Token':os.environ['MCP_JS_DEMO_TOKEN']}) as http:
            async with streamable_http_client(url,http_client=http) as streams:
                async with ClientSession(streams[0],streams[1]) as session:
                    await session.initialize()
                    return await investigation_loop(model,session,alert,prompt,emit)


### Concurrency and stop behavior

Start schedules an **async I/O task** and returns control to the kernel. Other cells can execute while the task awaits model/MCP responses. This is not two Python cells executing CPU work in parallel: a long synchronous cell can delay the task and Stop handling. Keep presentation cells short. Operator script buttons are synchronous and may occupy the kernel while they run; avoid running them alongside the investigation.

Stop requests cancellation; it does not undo completed external effects. Rerunning the control cell first cancels and awaits its previous task, then disables the old controls. If cleanup exceeds five seconds, it refuses a replacement task; inspect state and restart the kernel. Repeated Start clicks cannot create overlapping runs. Traces remain explicitly visible in the output widget; clear outputs before sharing.

Tested in actual JupyterLab with a clearly labeled delayed offline fixture, not paid inference or live MCP. See evidence/notebook-ui-checks.md.

In [ ]:
import ipywidgets as w
# Rerun cleanup: never abandon a background task or leave old buttons active.
previous_task = globals().get('agent_task')
if previous_task is not None and not previous_task.done():
    previous_task.cancel()
    try:
        await asyncio.wait_for(asyncio.shield(previous_task), timeout=5)
    except asyncio.CancelledError:
        pass
    except asyncio.TimeoutError:
        raise RuntimeError('Previous task did not stop; interrupt/restart kernel before rerunning')
for old_control in globals().get('agent_controls', []):
    old_control.disabled = True
if globals().get('agent_ui') is not None:
    agent_ui.close()

alert_input=w.Textarea(value=json.dumps(ALERT),description='Alert:',layout=w.Layout(width='100%',height='90px'))
approved=w.Checkbox(value=False,description='Provider, synthetic data transfer and budget approved',layout=w.Layout(width='100%'))
start=w.Button(description='Start live agent');stop=w.Button(description='Stop agent',disabled=True)
agent_status=w.HTML(value='<b>Idle — no run started</b>')
agent_output=w.Output(); agent_task=None

async def launch_agent():
    try:
        def emit(value):
            text=json.dumps(value,indent=2)
            for key in ['AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','MCP_JS_DEMO_TOKEN']:
                secret=os.environ.get(key)
                if secret: text=text.replace(secret,'[REDACTED]')
            # Append explicitly; don't capture another cell's output while awaiting I/O.
            agent_output.append_stdout(text+'\n')
        await live_investigation(json.loads(alert_input.value),emit)
        agent_status.value='<b>Stopped normally — verify evidence before claiming completion</b>'
    except asyncio.CancelledError:
        agent_output.append_stdout('Cancelled. Inspect existing roots/effects before retry.\n')
        agent_status.value='<b>Cancelled — completed side effects remain</b>'
    except Exception as error:
        agent_output.append_stdout(type(error).__name__+': live run stopped; inspect the local request, no automatic retry.\n')
        agent_status.value='<b>Failed — no automatic retry</b>'
    finally:
        start.disabled=False;stop.disabled=True

def start_agent(_):
    global agent_task
    if not approved.value or (agent_task and not agent_task.done()): return
    approved.value=False
    start.disabled=True;stop.disabled=False
    agent_status.value='<b>Running — async I/O task; other cells may run</b>'
    agent_output.append_stdout('New manually approved run.\n')
    agent_task=asyncio.create_task(launch_agent())

def stop_agent(_):
    if agent_task and not agent_task.done():
        agent_status.value='<b>Cancellation requested</b>'
        agent_task.cancel()

start.on_click(start_agent);stop.on_click(stop_agent)
agent_controls=[start,stop,approved]
agent_ui=w.VBox([alert_input,approved,w.HBox([start,stop]),agent_status,agent_output])
display(agent_ui)


## 03–12 · Explain Compose while the agent works

Show the **actual generated** `.state/compose.json` beside this reference. Full topology: query → grafana-mcp → grafana → prometheus/loki, browser in the shared network namespace, policy files, COW object storage, node1/node2/node3 and publisher. The reduced investigation profile is single-voter; never describe it as three-voter availability.

If the agent finishes early keep evidence visible. If slow, expose actual status; no synthetic progress. The virtual COW tree is not Grafana’s host volume: publication is an explicit export.

In [ ]:
reference=json.loads((HERE/'compose.astronomy.json').read_text())
for name in ['query','grafana-mcp','browser','node1','node2','node3','objects','publisher']:
    service=reference['services'][name]
    # Only checked-in reference; do not dump live environment/secrets.
    print(name, json.dumps({k:service[k] for k in ['networks','network_mode','volumes'] if k in service},indent=2))
display(SVG(filename=str(HERE/'diagrams/flow.svg')))

### Deterministic fallback controls — manual clicks only

These call existing scripts, **not a model**. Seed only once during preparation. Query/log buttons use their existing script windows; for correlation use the explicit common window below through the agent's bound run_js. Both queries must return nonempty data before drawing conclusions.

In [ ]:
panel(['status','alert','query','logs'])

In [ ]:
from datetime import datetime, timezone, timedelta
end=datetime.now(timezone.utc).replace(microsecond=0); start=end-timedelta(minutes=5)
window={'start':start.isoformat().replace('+00:00','Z'),'end':end.isoformat().replace('+00:00','Z')}
metric_args={'datasourceUid':'demo-prometheus','expr':'sum by (service_name,status_code) (rate(traces_span_metrics_calls_total{service_name="checkout"}[2m]))','startTime':window['start'],'endTime':window['end'],'stepSeconds':5,'queryType':'range'}
log_args={'datasourceUid':'demo-logs','logql':'{service_name=~"checkout|payment"}','startRfc3339':window['start'],'endRfc3339':window['end'],'limit':50}
assert metric_args['startTime']==log_args['startRfc3339']
print('Proposed common UTC window, NOT query results:',window)
print(json.dumps({'query_prometheus':metric_args,'query_loki_logs':log_args},indent=2))

## 12–18 · Preview handoff and policy

Use the complete canonical prompt. First publication is a selected single-root preview; after merge, explicitly select the merged root and replace it. `dashboard` is a deterministic fixture generator, not the agent's dashboard. For an agent-created file, use the existing root selection command documented in README before Publish.

Require the publisher's root/hash receipt, loaded matching dashboard panels, then real browser graph data and the user link. API success alone is insufficient. Policy test currently writes `/work/allowed/notes.txt`, denies `/work/denied/secret.txt`, and verifies base readback; the agent's separate analysis policy permits scratch/dashboard paths. Do not call an arbitrary runtime exception a policy denial.

In [ ]:
panel(['policy','dashboard'])
panel(['select-root','publish'])

## 18–28 · Fork, isolate, merge, replace

In the Pi terminal: `/demo-init` before investigation, then `/paired-fork` at the current leaf. Read `branch_info` and record both session IDs. In the child, `/paired-merge PARENT_SESSION_ID` merges filesystem changes into the child; conversation histories remain separate. The adapter now uses the direct child's recorded fork root even when the parent wrote after init. Unrelated unequal bases fail closed.

No model or interactive extension execution is claimed by `pair-test`: it exercises real Pi SessionManager and MCP with explicit synthetic fixture messages. The branch/merge/conflict controls are deterministic engine checks. Preserve actual B/A/T/M IDs and show conflict/CAS rejection. Explicitly select the merged dashboard root before publishing the replacement; don't reseed.

In [ ]:
display(SVG(filename=str(HERE/'diagrams/branches.svg')))
panel(['pair-test','branches','merge','conflict','publish'])

## 28–35 · Controlled node failure and recovery

**Manual only.** Require the three-voter phase, healthy owned nodes, existing seed, and restorable volumes. Confirm the exact project ID and one action at a time. No AWS control appears here. `outage` stops/restarts the owned leader; `quorum` restores attempted stopped nodes even if a later stop fails. Do not retry uncertain writes before inspecting the same label. Object storage is a single fixture, not replicated storage HA.

Stop after 30 seconds of live troubleshooting; keep the six-minute Q&A. If using staged memory profiles, show dated cluster evidence instead of implying simultaneous workload availability.

In [ ]:
panel(['status','outage','quorum'])

## 35–39 · Tradeoffs, reset and fallback

Reset moves the demo label to B; it does not undo active mounts, heaps, Grafana publication, Pi history or external calls. It is optional after evidence capture, not a recovery shortcut. The reset control below is label-only; volume deletion is deliberately absent. For operator fault cleanup use the README operator procedure and verify checkout metrics/browser before reinjecting. Do not run `demo.py test` mid-talk.

Fallback: retain exact query/root receipts and clearly dated records in evidence/. Earlier synthetic-fixture Grafana checks are not Astronomy success. [verification.md](verification.md) distinguishes executed checks and pending work. No live outputs are embedded here.

In [ ]:
panel(['reset'])

In [ ]:
display(Markdown((HERE/'verification.md').read_text()))

## 39–45 · Q&A

What survives a crash? Can a timed-out write commit later? Why is the COW tree not a host mount? What is the merge base? What does policy denial prove? What did Robert personally decide, and what is merely inferred from implementation?

Use exact source links in architecture-reference.md and slide notes. End with the actual dashboard URL and provenance, or the precise incomplete handoff check. The full runtime and paid model loop were **not executed during notebook preparation**.